# 02 — Data Cleaning and Transformation

Covers **Section 7 of the report — Data Primary Cleaning and Transformation**.

Notebook 01 found the problems. This notebook fixes them, one numbered decision at a time.

Each decision follows the same shape:

> **What we found** → **What we decided, and why** → **The code, with a count**

Several are genuine judgement calls where a reasonable person could choose otherwise. Those are
argued rather than asserted — a decision you cannot defend is worse than one someone disagrees
with.

**Outputs:** `transactions_clean.csv`, `returns.csv`, `baskets.csv`, `sku_weekly.csv`,
`absence_candidates.csv`, `cleaning_ledger.csv`

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW  = ROOT / "data" / "raw" / "uci"
PROC = ROOT / "data" / "processed"; PROC.mkdir(parents=True, exist_ok=True)

ledger = []
def log(step, decision, before, after):
    ledger.append({"step": step, "decision": decision, "rows_before": before,
                   "rows_after": after, "rows_removed": before-after})
    pct = 100*(before-after)/before if before else 0
    print(f"[{step}] {decision}")
    print(f"        {before:,} -> {after:,}   ({before-after:,} removed, {pct:.2f}%)")

src = RAW / "online_retail_II.xlsx"
sheets = pd.ExcelFile(src).sheet_names
parts = []
for s in sheets:
    p = pd.read_excel(src, sheet_name=s); p["source_sheet"] = s
    parts.append(p)
df = pd.concat(parts, ignore_index=True)
df.columns = [x.strip().replace(" ", "_") for x in df.columns]
N_RAW = len(df)

df["StockCode"]   = df["StockCode"].astype(str).str.strip().str.upper()
df["Description"] = df["Description"].astype(str).str.strip()
df["Invoice"]     = df["Invoice"].astype(str).str.strip()

print(f"raw lines loaded: {N_RAW:,}")

raw lines loaded: 1,067,371


## Decision 1a — Remove cross-sheet duplicates

**What we found.** Duplicate-flagged rows cluster overwhelmingly in **December 2010** — around
45,000 against 400–2,700 in every other month. The two sheets are *Year 2009-2010* and
*Year 2010-2011*, and both contain that month.

**What we decided.** Where the same transaction key appears in **both** sheets, keep one copy.

**Why.** These are not two events, they are one event published twice. Left in, every unit of
December 2010 demand would be counted twice — inflating that month's revenue, distorting the
seasonal peak the forecasting depends on, and doubling the support of any association rule
involving those baskets.

In [2]:
KEY = ["Invoice", "StockCode", "Quantity", "InvoiceDate", "Price"]

flagged = df.duplicated(subset=KEY, keep=False)
by_month = (df[flagged].groupby(pd.to_datetime(df.loc[flagged,"InvoiceDate"]).dt.to_period("M")).size())
print("duplicate-flagged rows, worst months:")
print(by_month.sort_values(ascending=False).head(4).to_string())

sheets_per_key = df.groupby(KEY, dropna=False)["source_sheet"].transform("nunique")
cross = (sheets_per_key > 1) & df.duplicated(subset=KEY, keep="first")

n = len(df)
df = df[~cross]
log("D1a", "remove cross-sheet duplicates (Dec 2010 published in both sheets)", n, len(df))

duplicate-flagged rows, worst months:
InvoiceDate
2010-12    45381
2010-11     2747
2011-11     2643
2010-10     1636
Freq: M
[D1a] remove cross-sheet duplicates (Dec 2010 published in both sheets)
        1,067,371 -> 1,044,527   (22,844 removed, 2.14%)


## Decision 1b — KEEP within-sheet duplicates

**What we found.** After removing the cross-sheet overlap, tens of thousands of identical lines
remain *inside* individual sheets.

**What we decided.** Keep them.

**Why.** Two identical lines on one invoice is ordinary retail behaviour — the same product
entered twice on an order. And critically, `InvoiceDate` in this dataset is recorded **per
invoice, not per line**, so every line on an order shares a timestamp by construction. Identical
timestamps are therefore expected, not evidence of an error.

**Why this is arguable.** Some of these may genuinely be double-entry mistakes. But the
asymmetry of the error matters: removing them deletes real demand and understates the affected
SKUs, which propagates into their reorder points. Keeping a small number of duplicated lines
inflates demand slightly; deleting real ones causes stockouts in the recommendation. Given the
choice, over-stating demand slightly is the safer failure.

In [3]:
within = df.duplicated(subset=KEY, keep=False).sum()
print(f"within-sheet duplicate lines KEPT: {within:,}")

ex = df[df.duplicated(subset=KEY, keep=False)].sort_values(KEY).head(4)
print("\nexample — same invoice, same product, two lines:")
print(ex[["Invoice","StockCode","Description","Quantity","Price"]].to_string(index=False, max_colwidth=32))

within-sheet duplicate lines KEPT: 22,200

example — same invoice, same product, two lines:
Invoice StockCode                     Description  Quantity  Price
 489517     21491 SET OF THREE VINTAGE GIFT WRAPS         1   1.95
 489517     21491 SET OF THREE VINTAGE GIFT WRAPS         1   1.95
 489517     21821 GLITTER STAR GARLAND WITH BELLS         1   3.75
 489517     21821 GLITTER STAR GARLAND WITH BELLS         1   3.75


## Decision 2 — Separate cancellations into a returns table

**What we found.** 8,292 invoices begin with `C`, covering ~19,000 lines, almost all with
negative quantities.

**What we decided.** Move them to `returns.csv`. Do not delete them.

**Why.** A cancellation is demand being *reversed*; it cannot sit in a sales table used to
forecast demand or mine baskets. But it is genuine information about return behaviour, and
secondary objective 11 analyses returns as a demand-quality signal. Deleting them would throw
away a whole line of analysis to save one filter.

In [4]:
is_cancel = df["Invoice"].str.upper().str.startswith("C")
returns = df[is_cancel].copy()
print(f"cancellation invoices: {returns['Invoice'].nunique():,}")
print(f"cancellation lines   : {len(returns):,}")
print(f"most-returned products:")
print(returns.groupby("Description")["Quantity"].sum().nsmallest(6).to_string())

n = len(df)
df = df[~is_cancel]
log("D2", "move cancellations to returns.csv (kept, not deleted)", n, len(df))

cancellation invoices: 8,292
cancellation lines   : 19,165
most-returned products:
Description
PAPER CRAFT , LITTLE BIRDIE           -80995
MEDIUM CERAMIC TOP STORAGE JAR        -74494
ROTATING SILVER ANGELS T-LIGHT HLDR    -9381
SET/6 FRUIT SALAD PAPER CUPS           -7140
SET/6 FRUIT SALAD  PAPER PLATES        -7008
Manual                                 -5444
[D2] move cancellations to returns.csv (kept, not deleted)
        1,044,527 -> 1,025,362   (19,165 removed, 1.83%)


## Decision 3 — Exclude service codes by explicit list, never by pattern

**What we found.** 63 stock codes do not follow the 5-digit product convention. Some are
services — `POST`, `DOT`, `M` (manual), `BANK CHARGES`, gift vouchers, `TEST001`. **Others are
real products** — `DCGS0058` is *MISO PRETTY GUM*, `DCGS0066N` is *NAVY CUDDLES DOG HOODIE*.

**What we decided.** Exclude a **named list** of service codes. Never a regular expression.

**Why this is the most consequential decision in the notebook.** A pattern such as
`^\d{5}[A-Z]*$` looks reasonable and would remove all 63 — silently deleting 37 genuine SKUs
along with the postage. Nothing in the output would reveal it: the row count would simply be
slightly lower, and those products would be absent from every association rule, every ABC
class, and every reorder recommendation.

Leaving service codes *in* is equally wrong in the other direction — postage would be assigned
a reorder point, and "Manual" would appear in association rules as though customers bought it.

In [5]:
SERVICE_CODES = {"POST","DOT","C2","M","D","S","BANK CHARGES","ADJUST","ADJUST2",
                 "AMAZONFEE","CRUK","B","TEST001","TEST002","PADS"}
is_voucher = df["StockCode"].str.startswith("GIFT_0001")
is_service = df["StockCode"].isin(SERVICE_CODES) | is_voucher

print("REMOVED as services:")
print(df[is_service].groupby("StockCode").size().sort_values(ascending=False).head(10).to_string())

n = len(df)
df = df[~is_service]
log("D3", "remove service codes, postage, vouchers and test rows", n, len(df))

kept = df.loc[~df["StockCode"].str.match(r"^\d{5}[A-Z]*$")]
print(f"\nKEPT as genuine products despite non-standard codes: {kept['StockCode'].nunique()}")
print(kept.drop_duplicates('StockCode')[["StockCode","Description"]].head(8).to_string(index=False, max_colwidth=40))

REMOVED as services:
StockCode
POST            1858
DOT             1422
M                868
C2               270
ADJUST            36
BANK CHARGES      34
GIFT_0001_20      29
GIFT_0001_30      29
PADS              18
GIFT_0001_10      16
[D3] remove service codes, postage, vouchers and test rows
        1,025,362 -> 1,020,724   (4,638 removed, 0.45%)

KEPT as genuine products despite non-standard codes: 37
StockCode                  Description
 DCGS0058             MISO PRETTY  GUM
 DCGS0068            DOGS NIGHT COLLAR
 DCGS0004   HAYNES CAMPER SHOULDER BAG
 DCGS0076 SUNJAR LED NIGHT NIGHT LIGHT
 DCGS0003          BOXED GLASS ASHTRAY
 DCGS0072       CAT CAMOUFLAGUE COLLAR
 DCGS0044      HANDZ-OFF CAR FRESHENER
DCGS0066N      NAVY CUDDLES DOG HOODIE


## Decision 4 — Impossible values

**What we found.** ~3,400 lines with non-positive quantity that are *not* cancellations, and
~2,600 with zero or negative price.

**What we decided.** Remove both.

**Why.** A negative quantity outside a cancellation is a stock write-off or a manual
adjustment — a warehouse event, not a customer buying something. A zero price is a giveaway or
a data error; either way it is not a sale and would drag every revenue-based calculation
downward, including the ABC classification that the entire stocking policy rests on.

In [6]:
n = len(df); bad = (df["Quantity"] <= 0).sum()
df = df[df["Quantity"] > 0]
log("D4", f"drop non-positive quantity ({bad:,} write-offs and adjustments)", n, len(df))

n = len(df); bad = (df["Price"] <= 0).sum()
df = df[df["Price"] > 0]
log("D4", f"drop zero or negative price ({bad:,} gifts and errors)", n, len(df))

n = len(df)
df = df[df["Description"].notna() & (df["Description"].str.lower() != "nan")]
log("D4", "drop rows with no product description", n, len(df))

[D4] drop non-positive quantity (3,393 write-offs and adjustments)
        1,020,724 -> 1,017,331   (3,393 removed, 0.33%)
[D4] drop zero or negative price (2,580 gifts and errors)
        1,017,331 -> 1,014,751   (2,580 removed, 0.25%)
[D4] drop rows with no product description
        1,014,751 -> 1,014,751   (0 removed, 0.00%)


## Decision 5 — Derived fields

Revenue, calendar parts, and one canonical product name per SKU. Descriptions vary in spelling
between rows for the same code, so the most frequent spelling is taken as the label — otherwise
the same product appears under several names in the dashboard.

In [7]:
df["revenue"] = df["Quantity"] * df["Price"]
df["date"]  = pd.to_datetime(df["InvoiceDate"]).dt.normalize()
df["week"]  = df["date"].dt.to_period("W").dt.start_time
df["month"] = df["date"].dt.to_period("M").astype(str)
df["dow"]   = df["date"].dt.dayofweek
df["is_uk"] = (df["Country"] == "United Kingdom").astype(int)

name = (df.groupby("StockCode")["Description"]
        .agg(lambda s: s.value_counts().index[0]).rename("product"))
df = df.merge(name, on="StockCode", how="left")

print(f"rows       : {len(df):,}")
print(f"SKUs       : {df['StockCode'].nunique():,}")
print(f"invoices   : {df['Invoice'].nunique():,}")
print(f"period     : {df['date'].min().date()} to {df['date'].max().date()}")
print(f"revenue    : £{df['revenue'].sum():,.0f}")

rows       : 1,014,751
SKUs       : 4,724
invoices   : 39,516
period     : 2009-12-01 to 2011-12-09
revenue    : £19,699,733


## Decision 6 — The basket table

Association mining needs invoices containing two or more distinct SKUs. Single-item invoices
carry no co-purchase information and are excluded from this table only — they remain in the
sales table, because they are still demand.

In [8]:
bsize = df.groupby("Invoice")["StockCode"].nunique()
multi = bsize[bsize > 1].index
baskets = df[df["Invoice"].isin(multi)][["Invoice","StockCode","product"]].drop_duplicates()

print(f"multi-item invoices : {len(multi):,}  ({100*len(multi)/len(bsize):.1f}%)")
print(f"basket lines        : {len(baskets):,}")
print(f"median basket       : {bsize[multi].median():.0f} SKUs")
print(f"largest basket      : {bsize.max():,} SKUs  <- outlier, check before mining")

multi-item invoices : 36,308  (91.9%)
basket lines        : 988,828
median basket       : 17 SKUs
largest basket      : 1,106 SKUs  <- outlier, check before mining


## Decision 7 — Weekly demand per SKU

Weekly rather than daily: daily demand for most SKUs is mostly zeros, which makes both
forecasting and volatility measurement unstable. Weekly aggregation keeps enough resolution for
reorder decisions while producing a series a model can actually work with.

In [9]:
weekly = (df.groupby(["StockCode","week"])
          .agg(units=("Quantity","sum"), revenue=("revenue","sum"),
               orders=("Invoice","nunique")).reset_index())
all_weeks = pd.date_range(weekly["week"].min(), weekly["week"].max(), freq="W-MON")
wps = weekly.groupby("StockCode").size()

print(f"SKU-week rows      : {len(weekly):,}")
print(f"weeks in period    : {len(all_weeks)}")
print(f"median SKU appears : {wps.median():.0f} weeks")
print(f"SKUs with 26+ weeks: {(wps>=26).sum():,}   <- forecastable")
print(f"SKUs with under 8  : {(wps<8).sum():,}   <- XYZ will classify these as Z")

SKU-week rows      : 196,242
weeks in period    : 106
median SKU appears : 34 weeks
SKUs with 26+ weeks: 2,810   <- forecastable
SKUs with under 8  : 664   <- XYZ will classify these as Z


## Decision 8 — Absence episodes for the pull-through test

**What we decided.** A qualifying episode requires at least **8 weeks of selling history**,
a gap of at least **3 consecutive weeks**, and at least **4 weeks of sales afterwards**.

**Why each threshold exists:**

- *8 weeks before* — the SKU must have an established pattern, otherwise its "normal" rate is
  undefined and the comparison is meaningless
- *3 weeks gap* — long enough to be a genuine absence rather than a quiet fortnight
- *4 weeks after* — **the critical one.** Requiring the item to come back is what distinguishes
  a stockout from a **discontinued line**. Without it, every deleted product in the catalogue
  would be counted as a stockout

These thresholds are choices, not facts. They should be varied in the modelling notebook to
confirm the result is not an artefact of where the lines were drawn.

In [10]:
MIN_BEFORE, MIN_GAP, MIN_AFTER = 8, 3, 4
episodes = []
for sku, g in weekly.groupby("StockCode"):
    wk = set(g["week"])
    if len(wk) < MIN_BEFORE + MIN_AFTER:
        continue
    present = pd.Series(all_weeks.isin(list(wk)), index=all_weeks)
    grp = (present != present.shift()).cumsum()
    for _, run in present.groupby(grp):
        if run.iloc[0] or len(run) < MIN_GAP:
            continue
        before = present.loc[:run.index[0]].iloc[:-1]
        after  = present.loc[run.index[-1]:].iloc[1:]
        if before.sum() >= MIN_BEFORE and after.sum() >= MIN_AFTER:
            episodes.append({"StockCode":sku, "gap_start":run.index[0], "gap_end":run.index[-1],
                             "gap_weeks":len(run), "weeks_before":int(before.sum()),
                             "weeks_after":int(after.sum())})

absence = pd.DataFrame(episodes,
    columns=["StockCode","gap_start","gap_end","gap_weeks","weeks_before","weeks_after"])
if len(absence):
    print(f"qualifying episodes: {len(absence):,}")
    print(f"distinct SKUs      : {absence['StockCode'].nunique():,}")
    print(f"median gap         : {absence['gap_weeks'].median():.0f} weeks")
    print(f"\nVERDICT: objective 4 is {'VIABLE' if absence['StockCode'].nunique()>=30 else 'TOO THIN'}")
else:
    print("qualifying episodes: 0")
    print("\nVERDICT: objective 4 cannot be tested — demote to secondary and say why")

qualifying episodes: 6,896
distinct SKUs      : 2,034
median gap         : 4 weeks

VERDICT: objective 4 is VIABLE


## Save and reconcile

The ledger must add up. If `raw − removed ≠ clean`, rows disappeared without being recorded,
and every figure in the report becomes untrustworthy.

In [11]:
df.to_csv(PROC/"transactions_clean.csv", index=False)
returns.to_csv(PROC/"returns.csv", index=False)
baskets.to_csv(PROC/"baskets.csv", index=False)
weekly.to_csv(PROC/"sku_weekly.csv", index=False)
absence.to_csv(PROC/"absence_candidates.csv", index=False)

led = pd.DataFrame(ledger)
led.loc[len(led)] = {"step":"TOTAL","decision":"raw lines -> clean sales lines",
                     "rows_before":N_RAW,"rows_after":len(df),"rows_removed":N_RAW-len(df)}
led.to_csv(PROC/"cleaning_ledger.csv", index=False)

print("RECONCILIATION")
print(f"  raw lines    {N_RAW:,}")
print(f"  removed      {N_RAW-len(df):,}  ({100*(N_RAW-len(df))/N_RAW:.1f}%)")
print(f"  clean sales  {len(df):,}")
print(f"  reconciles   {N_RAW - (N_RAW-len(df)) == len(df)}")
print(f"  ({len(returns):,} of those removals were moved to returns.csv, not discarded)\n")
led

RECONCILIATION
  raw lines    1,067,371
  removed      52,620  (4.9%)
  clean sales  1,014,751
  reconciles   True
  (19,165 of those removals were moved to returns.csv, not discarded)



,step,decision,rows_before,rows_after,rows_removed
0,D1a,remove cross-sheet duplicates (Dec 2010 publis...,1067371,1044527,22844
1,D2,"move cancellations to returns.csv (kept, not d...",1044527,1025362,19165
2,D3,"remove service codes, postage, vouchers and te...",1025362,1020724,4638
3,D4,"drop non-positive quantity (3,393 write-offs a...",1020724,1017331,3393
4,D4,"drop zero or negative price (2,580 gifts and e...",1017331,1014751,2580
5,D4,drop rows with no product description,1014751,1014751,0
6,TOTAL,raw lines -> clean sales lines,1067371,1014751,52620


## Result

Six tables, each feeding a specific objective:

| File | Feeds |
|---|---|
| `transactions_clean.csv` | everything |
| `returns.csv` | secondary objective 11 |
| `baskets.csv` | objectives 3 and 8 — association rules |
| `sku_weekly.csv` | objectives 5, 6, 7 — XYZ, forecasting, reorder points |
| `absence_candidates.csv` | objective 4 — the pull-through test |
| `cleaning_ledger.csv` | report section 7 |

**Three things to carry into the report:**

- **The two duplicate types are different problems.** One is a publishing artefact and must go;
  the other is genuine demand and must stay. A single `drop_duplicates()` call would have got
  one of them wrong.
- **The service-code exclusion is an explicit list on purpose.** The obvious pattern-based rule
  would have silently deleted 37 real products, and nothing in the output would have shown it.
- **Decisions 1b and 8 are the weakest links.** Keeping within-sheet duplicates and the choice
  of absence thresholds are both judgement calls. Both should be tested for sensitivity in the
  modelling notebook — a result that survives its own robustness check is worth far more than
  one that was never questioned.
